
# 03 — Feature Engineering

## Objectives
- Load the cleaned training, validation and test datasets.
- Define the target variable and predictor features.
- Inspect feature distributions and categorical values.
- Prepare consistent preprocessing for machine learning.
- Prevent data leakage by fitting preprocessing on training data only.

## Current scope
Greater London residential transactions, 2023–2025.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/processed")

train = pd.read_csv(DATA_DIR / "train_2023.csv", low_memory=False)
validation = pd.read_csv(DATA_DIR / "validation_2024.csv", low_memory=False)
test = pd.read_csv(DATA_DIR / "test_2025.csv", low_memory=False)

datasets = {
    "Train (2023)": train,
    "Validation (2024)": validation,
    "Test (2025)": test,
}

for name, data in datasets.items():
    print(f"\n--- {name} ---")
    print("Shape:", data.shape)
    print("Columns:", data.columns.tolist())
    print("Missing values in key columns:")
    display(
        data[
            ["price", "property_type", "old_new", "duration", "district",
             "sale_year", "sale_month"]
        ].isna().sum()
    )



--- Train (2023) ---
Shape: (90712, 22)
Columns: ['transaction_id', 'price', 'date', 'postcode', 'property_type', 'old_new', 'duration', 'paon', 'saon', 'street', 'locality', 'town', 'district', 'county', 'ppd_category', 'record_status', 'year', 'has_lookup_match', 'price_under_1000_flag', 'price_over_10m_flag', 'sale_year', 'sale_month']
Missing values in key columns:


price            0
property_type    0
old_new          0
duration         0
district         0
sale_year        0
sale_month       0
dtype: int64


--- Validation (2024) ---
Shape: (100522, 22)
Columns: ['transaction_id', 'price', 'date', 'postcode', 'property_type', 'old_new', 'duration', 'paon', 'saon', 'street', 'locality', 'town', 'district', 'county', 'ppd_category', 'record_status', 'year', 'has_lookup_match', 'price_under_1000_flag', 'price_over_10m_flag', 'sale_year', 'sale_month']
Missing values in key columns:


price            0
property_type    0
old_new          0
duration         0
district         0
sale_year        0
sale_month       0
dtype: int64


--- Test (2025) ---
Shape: (102629, 22)
Columns: ['transaction_id', 'price', 'date', 'postcode', 'property_type', 'old_new', 'duration', 'paon', 'saon', 'street', 'locality', 'town', 'district', 'county', 'ppd_category', 'record_status', 'year', 'has_lookup_match', 'price_under_1000_flag', 'price_over_10m_flag', 'sale_year', 'sale_month']
Missing values in key columns:


price            0
property_type    0
old_new          0
duration         0
district         0
sale_year        0
sale_month       0
dtype: int64


## Feature Selection

### Target variable
- `price`: recorded residential sale price in GBP.

### Initial predictor features
- `property_type`: flat, terraced, semi-detached or detached.
- `old_new`: new-build status.
- `duration`: freehold or leasehold.
- `district`: London borough/district.
- `sale_month`: month of transaction.

### Excluded columns
Transaction IDs, UPRNs, address fields, target-derived price flags and the original price from the predictor matrix are excluded to prevent identifier leakage and target leakage.

The year-based split is retained: 2023 training, 2024 validation and 2025 final testing.


In [2]:

# Define target and initial predictor columns

TARGET = "price"

FEATURES = [
    "property_type",
    "old_new",
    "duration",
    "district",
    "sale_month",
]

for name, data in datasets.items():
    print(f"\n--- {name} ---")

    for column in FEATURES:
        print(f"\n{column}: {data[column].nunique()} unique values")
        if data[column].nunique() <= 20:
            display(data[column].value_counts(dropna=False))
        elif column == "district":
            display(data[column].value_counts().head(10))

    print("\nTarget price summary:")
    display(data[TARGET].describe(
        percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
    ))

# Confirm the intended modelling columns exist
required = FEATURES + [TARGET]

for name, data in datasets.items():
    missing_columns = set(required) - set(data.columns)
    assert not missing_columns, (
        f"{name} is missing columns: {missing_columns}"
    )

print("\nFeature and target definitions verified.")



--- Train (2023) ---

property_type: 4 unique values


property_type
F    51319
T    23205
S    12154
D     4034
Name: count, dtype: int64


old_new: 2 unique values


old_new
N    81528
Y     9184
Name: count, dtype: int64


duration: 2 unique values


...


sale_month: 12 unique values


sale_month
3     19004
10     9160
8      8946
7      8847
9      8717
2      8385
1      7763
6      7610
11     7531
12     6689
5      6228
4      3749
Name: count, dtype: int64


Target price summary:


count    1.026290e+05
mean     6.793073e+05
std      8.546595e+05
min      8.000000e+03
1%       1.470000e+05
25%      3.785000e+05
50%      5.100000e+05
75%      7.250000e+05
99%      3.400000e+06
max      5.300000e+07
Name: price, dtype: float64


Feature and target definitions verified.


In [3]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Target and predictors
TARGET = "price"

CATEGORICAL_FEATURES = [
    "property_type",
    "old_new",
    "duration",
    "district",
]

NUMERIC_FEATURES = ["sale_month"]

FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

# Separate predictors and target
X_train = train[FEATURES].copy()
y_train = train[TARGET].copy()

X_val = validation[FEATURES].copy()
y_val = validation[TARGET].copy()

X_test = test[FEATURES].copy()
y_test = test[TARGET].copy()

# Preprocessing for categorical and numerical columns
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
])

# Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Validation and test only receive the learned transformations
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Training matrix:", X_train_processed.shape)
print("Validation matrix:", X_val_processed.shape)
print("Test matrix:", X_test_processed.shape)

print("\nTraining target:", y_train.shape)
print("Validation target:", y_val.shape)
print("Test target:", y_test.shape)

assert X_train_processed.shape[1] == X_val_processed.shape[1]
assert X_train_processed.shape[1] == X_test_processed.shape[1]

print("\nPASS: Feature preprocessing completed without fitting on validation or test data.")


Training matrix: (90712, 42)
Validation matrix: (100522, 42)
Test matrix: (102629, 42)

Training target: (90712,)
Validation target: (100522,)
Test target: (102629,)

PASS: Feature preprocessing completed without fitting on validation or test data.



## Feature Engineering Summary

### Target
- `price`: residential sale price in GBP.

### Selected predictors
- Property type
- New-build status
- Freehold/leasehold status
- District
- Sale month

### Preprocessing
- Categorical features: missing-value imputation and one-hot encoding.
- Numerical feature: median imputation and standardisation.
- Unknown categorical values are handled using `handle_unknown="ignore"`.
- The preprocessor was fitted on training data only and then applied to validation and test data.

### Output dimensions
- Training: 90,712 rows × 42 features.
- Validation: 100,522 rows × 42 features.
- Testing: 102,629 rows × 42 features.

### Limitations
- EPC features are not included.
- The model currently relies on a limited set of property characteristics and location information.
- The preprocessing stage has been verified; predictive performance has not yet been evaluated.
